In [2]:
import rioxarray as rxr
import numpy as np
from pathlib import Path
from joblib import Parallel, delayed
from tqdm import tqdm

def compute_transitions(tile_path: Path):
    raster_20 = rxr.open_rasterio(tile_path).squeeze()
    raster_21 = rxr.open_rasterio(tile_path.parent.parent / '2021' / tile_path.name).squeeze()
    raster_22 = rxr.open_rasterio(tile_path.parent.parent / '2022' / tile_path.name).squeeze()
    raster_23 = rxr.open_rasterio(tile_path.parent.parent / '2023' / tile_path.name).squeeze()
    raster_24 = rxr.open_rasterio(tile_path.parent.parent / '2024' / tile_path.name).squeeze()
    
    # if rasters have different shapes, reproject_match to the bounds of raster_20
    if raster_21.shape != raster_20.shape:
        raster_21 = raster_21.rio.reproject_match(raster_20)
    if raster_22.shape != raster_20.shape:
        raster_22 = raster_22.rio.reproject_match(raster_20)
    if raster_23.shape != raster_20.shape:
        raster_23 = raster_23.rio.reproject_match(raster_20)
    if raster_24.shape != raster_20.shape:
        raster_24 = raster_24.rio.reproject_match(raster_20)
    
    raster_stack = np.stack([raster_20, raster_21, raster_22, raster_23, raster_24], axis=0)
    transitions = np.zeros(raster_20.shape, dtype=np.uint8)
    for i in range(1, raster_stack.shape[0]):
        transitions += (raster_stack[i] != raster_stack[i - 1]).astype(np.uint8)
    transitions = raster_20.copy(data=transitions)
    transitions = transitions.rio.write_nodata(15).astype(np.uint8)
    
    out_path = tile_path.parent.parent / 'transitions' / tile_path.name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    transitions.rio.to_raster(out_path, compress='LZW', tiled=True)


tile_paths = list(Path('../runs/s2_out/2020/').rglob('*.tif'))
# Parallel(n_jobs=4)(delayed(compute_transitions)(tile_path) for tile_path in tqdm(tile_paths))

In [3]:
buffer = 16 # pixels to clip from each edge to remove edge artifacts

for folder in ['2020', '2021', '2022', '2023', '2024', 'transitions']:
    temp_out_path = Path('../runs/s2_out/') / folder / 'temp'
    temp_out_path.mkdir(parents=True, exist_ok=True)
    for file in tqdm(list(Path('../runs/s2_out/').glob(f'{folder}/*.tif'))):
        # clip around the edges to remove edge artifacts and save to temp folder
        raster = rxr.open_rasterio(file).squeeze()
        clipped = raster.isel(x=slice(buffer, -buffer), y=slice(buffer, -buffer))
        clipped.rio.to_raster(temp_out_path / file.name, compress='LZW', tiled=True)
    

100%|██████████| 50/50 [09:21<00:00, 11.23s/it]


In [ ]:
import rasterio
import geopandas as gpd
from rasterio.mask import mask
from rasterio.merge import merge
from joblib import Parallel, delayed
from dask.distributed import Client

try:
    client = Client()
except NameError:
    pass
client = Client()
display(client)


footprint_gdf = gpd.read_file(r'd:\chesapeake_bay_2022_2024edition\footprint.gpkg')
target_crs = footprint_gdf.crs

colormap = rasterio.open('../runs/s2_out/2020/17SMB.tif').colormap(1)

# for all the years and temp files in the temp folders, reproject them to the target CRS, and mask to the footprint

def reproject_and_mask(file):
        raster = rxr.open_rasterio(file, chunks=(1, 2048, 2048)).squeeze()
        raster_reprojected = raster.rio.reproject(target_crs)
        # mask to footprint
        raster_masked = raster_reprojected.rio.clip(footprint_gdf.geometry, all_touched=True)
        out_path = temp_out_path / file.name
        raster_masked.rio.to_raster(out_path, compress='LZW', tiled=True)
        # add colormap if applicable
        if folder != 'transitions':
            with rasterio.open(out_path, 'r+') as dst:
                dst.write_colormap(1, colormap)
                
for folder in ['2020', '2021', '2022', '2023', '2024', 'transitions']:
    temp_out_path = Path('../runs/s2_out/') / folder / 'temp_reprojected'
    temp_out_path.mkdir(parents=True, exist_ok=True)
    
    
    
    temp_files = list((Path('../runs/s2_out/') / folder / 'temp').glob('*.tif'))
    
    Parallel(n_jobs=32, backend='threading')(delayed(reproject_and_mask)(file) for file in tqdm(temp_files))    
    # for file in tqdm(temp_files):
    #     reproject_and_mask(file)


c:\Users\dh2306\projects\s2flow\.venv\Lib\site-packages\distributed\node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 61672 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:61672/status,
Dashboard: http://127.0.0.1:61672/status,Workers: 6
Total threads: 24,Total memory: 127.80 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:61675,Workers: 0
Dashboard: http://127.0.0.1:61672/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:61694,Total threads: 4
Dashboard: http://127.0.0.1:61695/status,Memory: 21.30 GiB
Nanny: tcp://127.0.0.1:61678,


100%|██████████| 50/50 [00:00<00:00, 4857.55it/s]
